In [45]:
import pandas as pd
import numpy as np
import os
import pickle
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [9]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
N_AA = len(AA_LIST)
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
idx_to_aa = {v: k for k, v in aa_to_idx.items()}

def min_max_norm(data):
    data_min = np.min(data)
    data_max = np.max(data)
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def magnitude_norm(data, data_min=0):
    data_max = np.max(np.abs(data))
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def predict_ok(predict_df, selected_seqno):
    pred = predict_df.loc[predict_df["Seq_no"] == selected_seqno, "Prediction"].values[0]
    true = predict_df.loc[predict_df["Seq_no"] == selected_seqno, "Label"].values[0]
    pred_0 = float(pred.split()[1].split(",")[0])
    pred_1 = float(pred.split()[3].split("}")[0])
    pred_class = 0
    if pred_1 > pred_0:
        pred_class = 1
    if pred_class == true:
        return True
    else:
        return False

def get_res_talk_optimized(attn_all_layers):
    # Slice arrays
    aa_attn_all_layers = attn_all_layers[:,:,1:-1,1:-1]
    cls_attn_all_layers = attn_all_layers[:,:,0,1:-1]
    
    # 1. Vectorized Min-Max Normalization
    # Calculate min and max along the last axis (L_SEQ), keeping dimensions 
    # so they broadcast back against the original array seamlessly.
    c_min = cls_attn_all_layers.min(axis=-1, keepdims=True)
    c_max = cls_attn_all_layers.max(axis=-1, keepdims=True)
    
    norm_cls = (cls_attn_all_layers - c_min) / (c_max - c_min)
    
    # 2. Einsum for simultaneous multiplication and summation
    # 'lhij' = aa_attn_all_layers: (layers, heads, L_SEQ_i, L_SEQ_j)
    # 'lhi'  = norm_cls:           (layers, heads, L_SEQ_i)
    # '->ij' = Output:             Sum over 'l' and 'h', leaving (L_SEQ_i, L_SEQ_j)
    sum_weighted_attn = np.einsum('lhij,lhi->ij', aa_attn_all_layers, norm_cls)
    
    return sum_weighted_attn

def process_single_attention(file_path):
    attn_all_layers = np.load(file_path)
    return get_res_talk_optimized(attn_all_layers)

def clipping_max_norm(data, cutoff_val=None, cutoff_percentile=99, return_cutoff=False):
    if cutoff_val is None:
        upper_limit = np.percentile(data, cutoff_percentile)
    else:
        upper_limit = cutoff_val
    clipped_matrix = np.clip(data, a_min=0, a_max=upper_limit)
    normalized_data = clipped_matrix / upper_limit
    if return_cutoff:
        return normalized_data, upper_limit
    else:
        return normalized_data

def process_single_attribution(file_path_0, file_path_1):
    attr_0 = np.load(file_path_0)
    norm_0 = magnitude_norm(attr_0)
    
    attr_1 = np.load(file_path_1)
    norm_1 = magnitude_norm(attr_1)
    
    return norm_0, norm_1

In [31]:
root_dir = "/pl/active/rdi_data/fahsai/PLM"
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]

attr_dir = os.path.join(root_dir, "results/attribution_maps/classification")
attn_dir = os.path.join(root_dir, "results/attention_maps/classification/full")
pred_dir = os.path.join(root_dir, "results/predictions/classification")
data_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis/data"
task_type = "classification"

info_df = pd.read_csv(os.path.join(data_dir, "input_info_VRC01_IC80.csv"))
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
seq_no_array = info_df['Seq_no'].values
true_labels = info_df["Label"].values
print(aligned_sequences.shape)
print(np.unique(true_labels, return_counts=True))

resno_df = pd.read_csv(os.path.join(data_dir, "selected_residues_IC80.csv"))
resno_array = resno_df["ResLabel"].values

N_MODEL = len(selected_models)
N_SEQ, _ = aligned_sequences.shape

all_prob_0 = np.zeros([N_MODEL, N_SEQ])
all_prob_1 = np.zeros([N_MODEL, N_SEQ])

for n, model_id in enumerate(selected_models):
    predict_df = pd.read_csv(os.path.join(pred_dir,"train_"+model_id+".csv"))
    for s, pred in enumerate(predict_df["Prediction"]):
        pred_0 = float(pred.split()[1].split(",")[0])
        pred_1 = float(pred.split()[3].split("}")[0])
        assert predict_df["Seq_no"].iloc[s] == info_df["Seq_no"].iloc[s]
        all_prob_0[n, s] = pred_0
        all_prob_1[n, s] = pred_1

mean_prob_0 = np.mean(all_prob_0, axis=0)
mean_prob_1 = np.mean(all_prob_1, axis=0)

pred_labels = np.zeros(N_SEQ, dtype=int)
pred_labels[mean_prob_1 > mean_prob_0] = 1

data_df = info_df.loc[info_df['Seq_no'].isin(seq_no_array[np.where(true_labels == pred_labels)])]
data_df.reset_index(drop=True, inplace=True)
aligned_sequences = np.array([list(seq) for seq in data_df['Sequence']])
class_label = data_df["Label"].values
seq_no_array = data_df['Seq_no'].values

print("--- After removing misprediction ---")
print(aligned_sequences.shape)
print(np.unique(class_label, return_counts=True))
N_SEQ, L_SEQ = aligned_sequences.shape

(889, 209)
(array([0, 1]), array([583, 306]))
--- After removing misprediction ---
(856, 209)
(array([0, 1]), array([569, 287]))


In [14]:
all_attr_array_0 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
all_attr_array_1 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
all_sum_attn_attr = np.zeros([N_MODEL, N_SEQ, L_SEQ, L_SEQ])
saved_model_cutoffs = {}

for n, model_name in enumerate(selected_models):
    print(f"Processing from {model_name}...")
    attr_file_paths_0 = [os.path.join(attr_dir, model_name, f"{no}_attribution_ig_class_0.npy") for no in data_df['Seq_no']]
    attr_file_paths_1 = [os.path.join(attr_dir, model_name, f"{no}_attribution_ig_class_1.npy") for no in data_df['Seq_no']]
    
    model_sum_attn_attr = np.zeros([N_SEQ, L_SEQ, L_SEQ])
    attn_file_paths = [os.path.join(attn_dir, model_name, f"{no}_attentions_raw.npy") for no in data_df['Seq_no']]
    
    with ThreadPoolExecutor(max_workers=8) as executor: 
        attr_results = list(tqdm(executor.map(process_single_attribution, attr_file_paths_0, attr_file_paths_1), total=N_SEQ))
        attn_results = list(tqdm(executor.map(process_single_attention, attn_file_paths), total=N_SEQ))
    
    for i, ((res_0, res_1), attn_res) in enumerate(zip(attr_results, attn_results)):
        all_attr_array_0[n, i] = res_0
        all_attr_array_1[n, i] = res_1
        model_sum_attn_attr[i] = attn_res
    normalized_data, cutoff_val = clipping_max_norm(model_sum_attn_attr, return_cutoff=True)
    all_sum_attn_attr[n] = normalized_data
    saved_model_cutoffs[model_name] = cutoff_val

 23%|██▎       | 196/856 [00:00<00:00, 1946.67it/s]

Processing from rep_5...


  3%|▎         | 26/856 [00:00<00:03, 259.89it/s]

Processing from rep_6...


  0%|          | 2/856 [00:00<00:49, 17.30it/s]

...

  1%|          | 9/856 [00:00<00:09, 89.72it/s]

Processing from rep_9...


  1%|          | 9/856 [00:00<00:09, 87.95it/s]

Processing from rep_10...


100%|██████████| 856/856 [01:16<00:00, 11.16it/s]


In [39]:
# Fields (H)

attr_1_array = all_attr_array_1.copy()
attr_0_array = all_attr_array_0.copy()

pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
conflict_mask = (pos_conflict_mask | neg_conflict_mask)
align_mask = ~conflict_mask

abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
abs_attr_score_array[conflict_mask] = 0
attr_score_array = abs_attr_score_array.copy()
# assign signs by class 1 attribution from each model
attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

mean_attr_score_array = np.mean(attr_score_array, axis=0)

h_map = {}
for i in range(L_SEQ):
    reslabel = resno_array[i]
    unique_aas = np.unique(aligned_sequences[:, i])
    for aa in unique_aas:
        mask = (aligned_sequences[:, i] == aa)
        h_map[(reslabel, aa)] = np.median(mean_attr_score_array[mask][:, i])

# Attention-derived communication maps (M)

residue_com_map = all_sum_attn_attr.copy()
mean_residue_com_map = np.mean(residue_com_map, axis=0)

In [44]:
# with open(root_dir+'/results/full/mean_attr_score_array.pkl', 'wb') as f:
#     pickle.dump(mean_attr_score_array, f)

# with open(root_dir+'/results/coupling/h_map.pkl', 'wb') as f:
#     pickle.dump(h_map, f)

# with open(root_dir+'/results/coupling/com_map.pkl', 'wb') as f:
#     pickle.dump(mean_residue_com_map, f)

# with open(root_dir+'/results/full/attn_cutoff.pkl', 'wb') as f:
#     pickle.dump(saved_model_cutoffs, f)

# with open(root_dir+'/results/full/aligned_sequences.pkl', 'wb') as f:
#     pickle.dump(aligned_sequences, f)

# with open(root_dir+'/results/full/seq_no_array.pkl', 'wb') as f:
#     pickle.dump(seq_no_array, f)

In [34]:
# with open(root_dir+'/results/coupling/h_map.pkl', 'rb') as c:
#     h_map = pickle.load(c)

# with open(root_dir+'/results/coupling/com_map.pkl', 'rb') as c:
#     mean_residue_com_map = pickle.load(c)

# with open(root_dir+'/results/full/aligned_sequences.pkl', 'rb') as c:
#     aligned_sequences = pickle.load(c)

In [77]:
all_sum_attn_attr = np.load("/scratch/alpine/fana6609/ML/PLM-Epistasis/results/coupling/all_sum_attn_attr.npy")
vulnerability = all_sum_attn_attr.copy()
mean_vulnerability = np.mean(vulnerability, axis=0)

all_attr_array_0 = np.load("/scratch/alpine/fana6609/ML/PLM-Epistasis/results/coupling/all_attr_array_0.npy")
all_attr_array_1 = np.load("/scratch/alpine/fana6609/ML/PLM-Epistasis/results/coupling/all_attr_array_1.npy")

attr_1_array = all_attr_array_1.copy()
attr_0_array = all_attr_array_0.copy()

pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
conflict_mask = (pos_conflict_mask | neg_conflict_mask)
align_mask = ~conflict_mask

abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
abs_attr_score_array[conflict_mask] = 0
attr_score_array = abs_attr_score_array.copy()
# assign signs by class 1 attribution from each model
attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

mean_attr_score_array = np.mean(attr_score_array, axis=0)

In [ ]:
# run src/fit_coupling.py to get epistatic_map.pkl

with open(root_dir+'/results/coupling/epistatic_map.pkl', 'rb') as c:
    loaded_data = pickle.load(c)
    epistatic_map = loaded_data['epistatic_map']

In [138]:
ATTR = np.zeros(N_SEQ)
ATTR_pred = np.zeros(N_SEQ)

for seq_idx in tqdm(range(N_SEQ), desc="Evaluating Sequences"):
    sequence = aligned_sequences[seq_idx]
    actual_attr = mean_attr_score_array[seq_idx]
    attn_matrix = mean_residue_com_map[seq_idx] # Shape: (Targets, Sources)

    E_pred = np.zeros(L_SEQ)

    # We iterate through the sequence, predicting the attribution for each TARGET
    for target_idx in range(L_SEQ):
        target_reslabel = resno_array[target_idx]
        target_aa = sequence[target_idx]
        
        # 1. Get the Baseline (H)
        h = h_map.get((target_reslabel, target_aa), 0.0)
        
        # 2. Calculate the Sum of Epistatic Couplings (C)
        c_total = 0.0
        
        for src_idx in range(L_SEQ):
            if src_idx == target_idx:
                continue # Skip self unless explicitly modeled
                
            src_reslabel = resno_array[src_idx]
            src_aa = sequence[src_idx]
            
            # Target (Query) paying attention to Source (Key)
            attn_val = attn_matrix[target_idx, src_idx]
            
            # Correct Key Order: Target First, Source Second
            w, b_conf, p_val, s_count = epistatic_map.get(
                (target_reslabel, target_aa, src_reslabel, src_aa), (0.0, 0.0, 1.0, 0)
            )
            
            # Apply your statistical filter
            if p_val < pval_cutoff:
                c_total += (attn_val * w)
                
        # The predicted attribution for this target residue
        E_pred[target_idx] = h + c_total
        
    # Sum the global attribution for the entire sequence
    ATTR[seq_idx] = actual_attr.sum()
    ATTR_pred[seq_idx] = E_pred.sum()

Evaluating Sequences: 100%|██████████| 856/856 [00:52<00:00, 16.18it/s]


In [141]:
# =====================================================================
# 3. STATISTICAL EVALUATION
# =====================================================================
print("\n--- Model Reconstruction Performance ---")
print(f"Spearman (Actual vs Pred): {spearmanr(ATTR, ATTR_pred)[0]:.4f} ({spearmanr(ATTR, ATTR_pred)[1]:.4f})")
print(f"Pearson  (Actual vs Pred): {pearsonr(ATTR, ATTR_pred)[0]:.4f} ({pearsonr(ATTR, ATTR_pred)[1]:.4f})")

print("\n--- Biological Ground Truth (pIC80) Correlation ---")
print(f"Spearman (pIC80 vs Actual ATTR): {spearmanr(data_df['pIC80'], ATTR)[0]:.4f} ({spearmanr(data_df['pIC80'], ATTR)[1]:.4f})")
print(f"Spearman (pIC80 vs Pred ATTR):  {spearmanr(data_df['pIC80'], ATTR_pred)[0]:.4f} ({spearmanr(data_df['pIC80'], ATTR_pred)[1]:.4f})")

print("\n--- Binary Neutralization Classification ---")
print(f"Accuracy (Actual ATTR): {accuracy_score(data_df['Label'], ATTR > 0):.4f} | Accuracy (Pred ATTR): {accuracy_score(data_df['Label'], ATTR_pred > -4):.4f}")
print(f"F1 Score (Actual ATTR): {f1_score(data_df['Label'], ATTR > 0):.4f} | F1 Score (Pred ATTR): {f1_score(data_df['Label'], ATTR_pred > -4):.4f}")
print(f"ROC AUC  (Actual ATTR): {roc_auc_score(data_df['Label'], ATTR):.4f} | ROC AUC  (Pred ATTR): {roc_auc_score(data_df['Label'], ATTR_pred):.4f}")


--- Model Reconstruction Performance ---
Spearman (Actual vs Pred): 0.9339 (0.0000)
Pearson  (Actual vs Pred): 0.9148 (0.0000)

--- Biological Ground Truth (pIC80) Correlation ---
Spearman (pIC80 vs Actual ATTR): 0.7234 (0.0000)
Spearman (pIC80 vs Pred ATTR):  0.7070 (0.0000)

--- Binary Neutralization Classification ---
Accuracy (Actual ATTR): 0.9930 | Accuracy (Pred ATTR): 0.9171
F1 Score (Actual ATTR): 0.9895 | F1 Score (Pred ATTR): 0.8782
ROC AUC  (Actual ATTR): 0.9999 | ROC AUC  (Pred ATTR): 0.9709
